Test I Submission. Built an MVP pipeline using OpenCV layout detection, Tesseract (LSTM), and simulated LLM post-processing. In the full GSoC project, this baseline will be upgraded to a fine-tuned TrOCR (ViT-TF) model.

In [1]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-spa
!pip install -q pdf2image opencv-python-headless pytesseract jiwer
print("✅ Libraries installed!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe

In [2]:
import os
os.makedirs('/content/dataset/pdfs', exist_ok=True)
os.makedirs('/content/dataset/images', exist_ok=True)
os.makedirs('/content/dataset/cropped_text', exist_ok=True)
print("✅ Folders created!")

✅ Folders created!


In [3]:
from pdf2image import convert_from_path
from PIL import Image
import glob
import os

Image.MAX_IMAGE_PIXELS = None

pdf_file = glob.glob('/content/dataset/pdfs/*.pdf')[0]
print(f"🔄 Extracting first 3 pages from {os.path.basename(pdf_file)}...")

images = convert_from_path(pdf_file, dpi=250, first_page=1, last_page=3)

for i, image in enumerate(images):
    image_path = f'/content/dataset/images/page_{i+1}.jpg'
    image.save(image_path, 'JPEG')

print("✅ Pages extracted safely!")

🔄 Extracting first 3 pages from PORCONES.23.5 - 1628.pdf...
✅ Pages extracted safely!


In [4]:
import cv2
import glob
import os

image_files = glob.glob('/content/dataset/images/*.jpg')
print("🔍 Cropping text blocks...")

for img_path in image_files:
    base_name = os.path.basename(img_path)
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Massive kernel to fuse paragraphs
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 50))
    dilated = cv2.dilate(thresh, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    padding = 20
    # Process the top 2 largest blocks (left and right columns)
    for i in range(min(2, len(contours))):
        x, y, w, h = cv2.boundingRect(contours[i])
        px = max(0, x - padding)
        py = max(0, y - padding)
        pw = min(img.shape[1] - px, w + (padding * 2))
        ph = min(img.shape[0] - py, h + (padding * 2))

        cropped_img = img[py:py+ph, px:px+pw]
        cv2.imwrite(f'/content/dataset/cropped_text/cropped_{i}_{base_name}', cropped_img)

print("✅ Text successfully isolated!")

🔍 Cropping text blocks...
✅ Text successfully isolated!


In [5]:
import pytesseract
import cv2
import glob

print("🚀 Running LSTM-based OCR...")
image_files = sorted(glob.glob('/content/dataset/cropped_text/*.jpg'))

raw_text_output = ""

for img_path in image_files:
    img = cv2.imread(img_path)
    text = pytesseract.image_to_string(img, lang='spa')
    raw_text_output += text + "\n"

print("✅ Raw OCR Output Generated:\n")
print(raw_text_output[:1000]) # Prints the first 1000 characters

🚀 Running LSTM-based OCR...
✅ Raw OCR Output Generated:

   
    
       
      

enn a e Retende doña Catalina,quedó An-
A PER tonio ha de fer condenado a G auien

"PS dofele.quitado el habito, e infignias

A
ZA
4 , , d 2)

he. 0) de la Orden,(ea entregado a la jufti.
| SN py: | ó , ' : : 4
JU e cia leglar, para qué fe:execute en él
SEA ES la pena de muerte, por los delitos q
=—__ a 5 s > ¿ eS | | 1 a
AAA cometió de aver con malos medios;
y tercerias, y promellas,y:alagos engañado a.D.Erácil*
: A Gd

 
  

    

a

    

2

fu Magéltad Jno ha quercllados y la hembra. (que esla

SY f h A"
, »

doña Erancifca calada oy con el matador) nolc pide,
han de premaleces a doña Catalina,g acuía por [er ma:
yor partes : ya Me. LD iIC 251 INS JR a
-. Pero éltedifcurlo repughaa la jurifprudencia, y ala
refolucion:mas alentada queay en clte Reyno.
- Lo:primero,que la.v iuda.tiene el primer logar en: la
aculaciomde la muerte del marido, y demas agravios
hechosa fuscafa,(1n qué los-hijos(quado huuie

In [6]:
def llm_historical_cleanup(raw_text):
    system_prompt = f"""
    You are an expert paleographer specializing in 17th-century Spanish typography.
    The following text is raw OCR output from a 1628 legal allegation (Porcones).

    Task:
    1. Fix OCR errors (e.g., the long 'ſ' misread as 'f').
    2. Remove isolated bookbinding catchwords.
    3. CRITICAL: Preserve the historical orthography exactly as it was in 1628. Do not modernize.

    Raw OCR Text:
    {raw_text[:1000]}
    """
    return system_prompt

print("✅ LLM Pipeline step defined. System Prompt:")
print(llm_historical_cleanup(raw_text_output))

✅ LLM Pipeline step defined. System Prompt:

    You are an expert paleographer specializing in 17th-century Spanish typography. 
    The following text is raw OCR output from a 1628 legal allegation (Porcones).
    
    Task:
    1. Fix OCR errors (e.g., the long 'ſ' misread as 'f').
    2. Remove isolated bookbinding catchwords.
    3. CRITICAL: Preserve the historical orthography exactly as it was in 1628. Do not modernize.
    
    Raw OCR Text:
       
    
       
      

enn a e Retende doña Catalina,quedó An-
A PER tonio ha de fer condenado a G auien

"PS dofele.quitado el habito, e infignias

A
ZA
4 , , d 2)

he. 0) de la Orden,(ea entregado a la jufti.
| SN py: | ó , ' : : 4
JU e cia leglar, para qué fe:execute en él
SEA ES la pena de muerte, por los delitos q
=—__ a 5 s > ¿ eS | | 1 a
AAA cometió de aver con malos medios;
y tercerias, y promellas,y:alagos engañado a.D.Erácil*
: A Gd

 
  

    

a

    

2

fu Magéltad Jno ha quercllados y la hembra. (que esla

SY f h A"
, 

In [7]:
import jiwer

# of the 1628 Porcones exactly as they typed it here:
ground_truth = """RctcndcdoñaCat,aliná,,'}ÜCdó An:
tonio ha de fer condenadoa qauicn
cloíelc,g~itádo
el habito,t infigrii~¡
de la·-Qrdcn,feen
a tregadoaJa•juíH~
•ciafeglar, paraqué fo·cxe,cuten
c: el
~~~iMJi' la penade muerte,po·,los dclicos:q
cometiodeaucr·-conmalosmcd1oí.
y tcrccdas,ypromcífas;y·aiagos
cngai-adoa E>.F,á·ci"""

# 2. This takes the first chunk of your Tesseract output
raw_ocr_output = raw_text_output[:len(ground_truth) + 100]

# 3. Paste what ChatGPT/Gemini just gave you here: actually I haven't used API and manually piplined with LLM due to deadline.
llm_cleaned_output = "PPretende doña Catalina, que don Antonio ha de ſer condenado a q auiendoſele quitado el habito, e inſignias de la Orden, ſea entregado a la juſticia ſeglar, para que ſe execute en él la pena de muerte, por los delitos q cometió de aver con malos medios, y tercerias, y promeſſas, y alagos engañado a D. Franciſ-"

try:
    raw_cer = jiwer.cer(ground_truth, raw_ocr_output)
    clean_cer = jiwer.cer(ground_truth, llm_cleaned_output)

    print(f"Metrics Evaluation (Character Error Rate):")
    print(f"1. Pre-LLM Pipeline (LSTM only): {raw_cer:.2%}")
    print(f"2. Post-LLM Pipeline: {clean_cer:.2%}")
    print(f"Improvement: {(raw_cer - clean_cer):.2%} reduction in errors.")
except ValueError:
    print("⚠️ Please paste the text into the variables above first!")

Metrics Evaluation (Character Error Rate):
1. Pre-LLM Pipeline (LSTM only): 73.05%
2. Post-LLM Pipeline: 42.53%
Improvement: 30.52% reduction in errors.
